In [ ]:
!wget -O ecg_data.zip "https://data.mendeley.com/public-api/zip/gwbz3fsgp8/download/1"
!unzip -q ecg_data.zip -d ecg_data
!find ecg_data -maxdepth 3 | head -30

--2026-08-14 20:26:54--  https://data.mendeley.com/public-api/zip/gwbz3fsgp8/download/1
Resolving data.mendeley.com... connected.
HTTP request sent, awaiting response... 302 Found -> following redirect to dataset storage
ecg_data.zip        100%[===================>] 379.21M  25.8MB/s    in 16s
2026-08-14 20:27:10 (24.3 MB/s) - 'ecg_data.zip' saved [397626879/397626879]

ecg_data/ECG Images of COVID-19 Patients (250)/...
ecg_data/ECG Images of Myocardial Infarction Patients (77)/...
ecg_data/ECG Images of Patient that have abnormal heart beats (548)/...
ecg_data/ECG Images of Patient that have History of MI (203)/...
ecg_data/Normal Person ECG Images (859)/...


In [ ]:
!ls -la ecg_data

total 96
drwxr-xr-x 7 root root  4096 Aug 14 20:27  .
drwxr-xr-x 1 root root  4096 Aug 14 20:27  ..
drwxr-xr-x 2 root root 12288 Nov 12  2020 'ECG Images of COVID-19 Patients (250)'
drwxr-xr-x 2 root root  4096 Nov 12  2020 'ECG Images of Myocardial Infarction Patients (77)'
drwxr-xr-x 2 root root 20480 Nov 12  2020 'ECG Images of Patient that have abnormal heart beats (548)'
drwxr-xr-x 2 root root 12288 Nov 12  2020 'ECG Images of Patient that have History of MI (203)'
drwxr-xr-x 2 root root 36864 Nov 12  2020 'Normal Person ECG Images (859)'


In [ ]:
import pandas as pd
import glob
import os

DATA_ROOT = "ecg_data"

ABNORMAL_FOLDERS = [
    "ECG Images of Patient that have abnormal heart beats (548)",
    "ECG Images of Myocardial Infarction Patients (77)",
    "ECG Images of Patient that have History of MI (203)",
]
NORMAL_FOLDERS = [
    "Normal Person ECG Images (859)",
]

rows = []
for folder in NORMAL_FOLDERS:
    for path in glob.glob(os.path.join(DATA_ROOT, folder, "*.jpg")):
        rows.append({"image_path": path, "label": 0})
for folder in ABNORMAL_FOLDERS:
    for path in glob.glob(os.path.join(DATA_ROOT, folder, "*.jpg")):
        rows.append({"image_path": path, "label": 1})

df = pd.DataFrame(rows)
print(df["label"].value_counts())
print(f"Total: {len(df)} images")

label
0    859
1    823
Name: count, dtype: int64
Total: 1682 images


In [ ]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
print(f"Train: {len(train_df)} ({train_df['label'].value_counts().to_dict()})")
print(f"Valid: {len(valid_df)} ({valid_df['label'].value_counts().to_dict()})")

Train: 1345 ({0: 687, 1: 658})
Valid: 337 ({0: 172, 1: 165})


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class ECGDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return image, label

train_loader = DataLoader(ECGDataset(train_df, transform), batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(ECGDataset(valid_df, transform), batch_size=32, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, labels shape: {labels.shape}")

Using device: cuda
Batch shape: torch.Size([32, 3, 224, 224]), labels shape: torch.Size([32])


In [ ]:
import copy
from sklearn.metrics import confusion_matrix

def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()
    total_loss = correct = total = 0
    with torch.set_grad_enabled(is_training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

model = models.resnet18(weights="IMAGENET1K_V1")
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

for epoch in range(1, 6):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_acc = run_epoch(model, valid_loader, criterion, optimizer=None)
    print(f"[Head only] Epoch {epoch}: train_acc={train_acc:.3f} valid_acc={valid_acc:.3f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s]


[Head only] Epoch 1: train_acc=0.694 valid_acc=0.769
[Head only] Epoch 2: train_acc=0.787 valid_acc=0.795
[Head only] Epoch 3: train_acc=0.799 valid_acc=0.828
[Head only] Epoch 4: train_acc=0.810 valid_acc=0.816
[Head only] Epoch 5: train_acc=0.810 valid_acc=0.822


In [ ]:
for name, param in model.named_parameters():
    if name.startswith("layer3") or name.startswith("layer4"):
        param.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-3},
])

best_valid_acc = 0.0
best_state = None

for epoch in range(1, 6):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_acc = run_epoch(model, valid_loader, criterion, optimizer=None)
    gap = train_acc - valid_acc
    is_best = valid_acc > best_valid_acc
    if is_best:
        best_valid_acc = valid_acc
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch}: train_acc={train_acc:.3f} | valid_acc={valid_acc:.3f} | gap={gap:.3f}" + ("  <-- new best, saved" if is_best else ""))

model.load_state_dict(best_state)
print(f"\nLoaded best model: valid_acc={best_valid_acc:.3f}")

Epoch 1: train_acc=0.816 | valid_acc=0.843 | gap=-0.027  <-- new best, saved
Epoch 2: train_acc=0.869 | valid_acc=0.843 | gap=0.026
Epoch 3: train_acc=0.888 | valid_acc=0.855 | gap=0.034  <-- new best, saved
Epoch 4: train_acc=0.923 | valid_acc=0.837 | gap=0.086
Epoch 5: train_acc=0.956 | valid_acc=0.852 | gap=0.105

Loaded best model: valid_acc=0.855


In [ ]:
model.eval()
all_probs = []
all_labels = []
with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        probs = torch.sigmoid(model(images).squeeze(1)).cpu()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.tolist())

import numpy as np
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

preds = (all_probs >= 0.5).astype(float)
tn, fp, fn, tp = confusion_matrix(all_labels, preds, labels=[0, 1]).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"Sensitivity (catching real Abnormal): {sensitivity:.3f}")
print(f"Specificity (correctly clearing Normal): {specificity:.3f}")
print(f"tn={tn}, fp={fp}, fn={fn}, tp={tp}")

Sensitivity (catching real Abnormal): 0.855
Specificity (correctly clearing Normal): 0.855
tn=147, fp=25, fn=24, tp=141


In [ ]:
torch.save(model.state_dict(), "ecg_classifier.pth")
from google.colab import files
files.download("ecg_classifier.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>